In [1]:
from autogen import ConversableAgent, initiate_chats
import os
import pprint

In [2]:
# llm configuration for the agent
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": os.environ.get("GROQ_API_KEY"),
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai",
        }
    ],
    "cache_seed": None,
}

In [3]:
# onboarding personal information agent definition
onboarding_personal_information_agent = ConversableAgent(
    name="Onboarding_Personal_Information_Agent",
    system_message=""" you are an helpful custemer onboarding agent,
    you work for a phone provider called ACME.
    Your job is to gather the customer's name and location.
    Do not ask for any other information, only ask about the customer's name and location.
    after the customer gives you thie nam and location, repeat them and thank the user, and ask the user to answer with TERMINATE to move on to the describing thier issue.
    """,
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content", "").lower(),
)

In [4]:
# onboarding Issue Agent definition
onboarding_issue_agent = ConversableAgent(
    name="Onboarding_Issue_Agent",
    system_message=""" you are an helpful custemer onboarding agent,
    you work for a phone provider called ACME.
    Your job is to gather the product the customer use and issue they currently have with the product.
    Do not ask for any other information.
    After the customer describes their issue, repeast it and add
    "please anser with 'TERMINATE" if I have correctly understood your issue." 
    """,
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content", "").lower(),
)

In [5]:
# customer engagement agent definition
customer_engagement_agent = ConversableAgent(
    name="Customer_Engagement_Agent",
    system_message=""" you are a custemer engagement agent.
    You work for a phone provider called ACME.
    Your job is to engage with the customer in a friendly manner and provide them with helpful information and topic preferences.
    This could include fun facts, jokes, or intersting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.
    """,
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content", "").lower(),
)

In [6]:
# coustomer proxy agent definition
customer_proxy_agent = ConversableAgent(
    name="Customer_Proxy_Agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
)

In [7]:
# chat orchetration
# create an empty list to store the conversation history
chats = []

In [8]:
# 1. onboarding personal information agent is the first point of contact, it will gather the customer's name and location
chats.append(
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": 
            "Hello, I'm here to help you solve any issue you have with our products. "
            "Could you tell me your name?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
        "summary_prompt" : "Return the customer information "
                             "into a JSON object only: "
                             "{'name': '', 'location': ''}",
        },
        "clear_history" : True
    }
)

In [9]:
# 2. after gathering the customer's name and location, onboarding issue agent will gather the issue the customer is having and with which product
chats.append(
    {
        "sender": onboarding_issue_agent,
        "recipient": customer_proxy_agent,
        "message": 
                "Great! Could you tell me what issue you're "
                "currently having and with which product?",
        "summary_method": "reflection_with_llm",
        "clear_history" : False
    }
)

In [10]:
# 3. after gathering the customer's issue, the customer engagement agent will 
# engage with the customer based on the information gathered from the previous 2 agents, 
# and provide them with helpful information and topic preferences to choose from while they wait for a human agent to assist them.
chats.append(
        {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "While we're waiting for a human agent to take over and help you solve "
        "your issue, can you tell me more about how you use our products or some "
        "topics interesting for you?",
        "max_turns": 2,
        "summary_method": "reflection_with_llm",
    }
)
chats

[{'sender': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ff9a2f90>,
  'recipient': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ffb851d0>,
  'message': "Hello, I'm here to help you solve any issue you have with our products. Could you tell me your name?",
  'summary_method': 'reflection_with_llm',
  'summary_args': {'summary_prompt': "Return the customer information into a JSON object only: {'name': '', 'location': ''}"},
  'clear_history': True},
 {'sender': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ff996c10>,
  'recipient': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ffb851d0>,
  'message': "Great! Could you tell me what issue you're currently having and with which product?",
  'summary_method': 'reflection_with_llm',
  'clear_history': False},
 {'sender': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ffb851d0>,
  'recipient': <autogen.agentchat.conversable_agent.ConversableAgent at 0x1b8ff997ed0>

In [11]:
chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
Onboarding_Personal_Information_Agent (to Customer_Proxy_Agent):

Hello, I'm here to help you solve any issue you have with our products. Could you tell me your name?

--------------------------------------------------------------------------------


c:\Users\02717461\OneDrive - Elanco\Desktop\GitHub\GenAI\.venv1\Lib\site-packages\autogen\agentchat\chat.py:71: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Customer_Proxy_Agent (to Onboarding_Personal_Information_Agent):

Raj

--------------------------------------------------------------------------------
[autogen.oai.client: 03-09 10:26:30] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Onboarding_Personal_Information_Agent (to Customer_Proxy_Agent):

Hello Raj, now I just need to know your location. Where are you located?

--------------------------------------------------------------------------------
Customer_Proxy_Agent (to Onboarding_Personal_Information_Agent):

India

--------------------------------------------------------------------------------
[autogen.oai.client: 03-09 10:26:40] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.

In [13]:
for chat_result in chat_results:
    pprint.pprint(chat_result.summary)

'{"name": "Raj", "location": "India"}'
('No further conversation took place to summarize. The user had an issue with '
 'their new TV not turning on.')
'No conversation took place to summarize.'
